# Classificatore basato su Support Vector Machine (SVM) linearie informazioni linguistiche non lessicali.

In questo notebook viene svolto il secondo punto del progetto, ossia: Classificatore basato su Support Vector Machine (SVM) lineari
e informazioni linguistiche non lessicali.




### 1) Import necessari

In [34]:
import pandas as pd
import numpy as np
import os
import zipfile
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

### 2) Creo un file .txt per ogni riga del dataframe, poi lo zippo per analizzarlo con profiling-UD

In [35]:
df_train = pd.read_csv("df_train.csv")
df_val = pd.read_csv("df_val.csv")
df_test = pd.read_csv("df_test.csv")


In [36]:
""""os.makedirs("train_docs", exist_ok=True)

for idx, row in df_train.iterrows():
    with open(f"train_docs/{idx}.txt", "w", encoding="utf-8") as f:
        f.write(row["text"])

with zipfile.ZipFile("train_docs.zip", "w") as zf:
    for idx in df_train.index:
        zf.write(f"train_docs/{idx}.txt")"""

'"os.makedirs("train_docs", exist_ok=True)\n\nfor idx, row in df_train.iterrows():\n    with open(f"train_docs/{idx}.txt", "w", encoding="utf-8") as f:\n        f.write(row["text"])\n\nwith zipfile.ZipFile("train_docs.zip", "w") as zf:\n    for idx in df_train.index:\n        zf.write(f"train_docs/{idx}.txt")'

In [37]:
"""os.makedirs("val_docs", exist_ok=True)
for idx, row in df_val.iterrows():
    with open(f"val_docs/{idx}.txt", "w", encoding="utf-8") as f:
        f.write(row["text"])
with zipfile.ZipFile("val_docs.zip", "w") as zf:
    for idx in df_val.index:
        zf.write(f"val_docs/{idx}.txt")

os.makedirs("test_docs", exist_ok=True)
for idx, row in df_test.iterrows():
    with open(f"test_docs/{idx}.txt", "w", encoding="utf-8") as f:
        f.write(row["text"])
with zipfile.ZipFile("test_docs.zip", "w") as zf:
    for idx in df_test.index:
        zf.write(f"test_docs/{idx}.txt")"""

'os.makedirs("val_docs", exist_ok=True)\nfor idx, row in df_val.iterrows():\n    with open(f"val_docs/{idx}.txt", "w", encoding="utf-8") as f:\n        f.write(row["text"])\nwith zipfile.ZipFile("val_docs.zip", "w") as zf:\n    for idx in df_val.index:\n        zf.write(f"val_docs/{idx}.txt")\n\nos.makedirs("test_docs", exist_ok=True)\nfor idx, row in df_test.iterrows():\n    with open(f"test_docs/{idx}.txt", "w", encoding="utf-8") as f:\n        f.write(row["text"])\nwith zipfile.ZipFile("test_docs.zip", "w") as zf:\n    for idx in df_test.index:\n        zf.write(f"test_docs/{idx}.txt")'

### 3) Import di train, validation e test set UD

In [38]:
train_ud = pd.read_csv("TRAINUD.csv", sep='\t')
val_ud = pd.read_csv("VALUD.csv", sep='\t')
test_ud = pd.read_csv("TESTUD.csv", sep='\t')

3.1) Train

In [39]:
# Mapping dal nome del file al doc_id

train_ud["doc_id"] = (
    train_ud["Filename"]
    .str.split("/")
    .str[-1]
    .str.replace(".conllu", "")
    .astype(int)
)

C:\Users\jacop\AppData\Local\Temp\ipykernel_8\4149085465.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_ud["doc_id"] = (


In [40]:
# Settiamo l'indice del DataFrame sui doc_id e selezioniamo solo le colonne delle feature

train_ud.set_index("doc_id", inplace=True)
feature_cols_t = [col for col in train_ud.columns if col != "Filename"]
train_ud = train_ud[feature_cols_t]

In [41]:
# Estraggo le label
labels_train = df_train.loc[train_ud.index, "label"].values

3.2) Validation

In [42]:
# Mapping dal nome del file al doc_id

val_ud["doc_id"] = (
    val_ud["Filename"]
    .str.split("/")
    .str[-1]
    .str.replace(".conllu", "")
    .astype(int)
)

C:\Users\jacop\AppData\Local\Temp\ipykernel_8\422538332.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  val_ud["doc_id"] = (


In [43]:
# Settiamo l'indice del DataFrame sui doc_id e selezioniamo solo le colonne delle feature

val_ud.set_index("doc_id", inplace=True)
feature_cols_v = [col for col in val_ud.columns if col != "Filename"]
val_ud = val_ud[feature_cols_v]

In [44]:
# Estraggo le label
labels_val = df_val.loc[val_ud.index, "label"].values

3.3) Test

In [45]:
# Mapping dal nome del file al doc_id

test_ud["doc_id"] = (
    test_ud["Filename"]
    .str.split("/")
    .str[-1]
    .str.replace(".conllu", "")
    .astype(int)
)

C:\Users\jacop\AppData\Local\Temp\ipykernel_8\3992318984.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_ud["doc_id"] = (


In [46]:
# Settiamo l'indice del DataFrame sui doc_id e selezioniamo solo le colonne delle feature

test_ud.set_index("doc_id", inplace=True)
feature_cols_tst = [col for col in test_ud.columns if col != "Filename"]
test_ud = test_ud[feature_cols_tst]

In [47]:
labels_test = df_test.loc[test_ud.index, "label"].values

In [48]:
# Allineo le colonne sul train test, dato che train ud potrebbe trovare numero di features differenti per file differenti

common_cols = train_ud.columns.intersection(val_ud.columns).intersection(test_ud.columns)
train_ud = train_ud[common_cols]
val_ud = val_ud[common_cols]
test_ud = test_ud[common_cols]

In [49]:
x_train = train_ud.values
y_train = labels_train

x_val = val_ud.values
y_val = labels_val

x_test = test_ud.values
y_test = labels_test

In [50]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_val = scaler.transform(x_val)
x_test = scaler.transform(x_test)

### 4) Addestramento del modello + validazione su validation e test set

In [51]:
clf = LinearSVC(max_iter=5000, random_state= 42)
clf.fit(x_train, y_train)

print("Validation:")
print(classification_report(y_val, clf.predict(x_val)))

Validation:
              precision    recall  f1-score   support

           0       0.94      0.94      0.94       500
           1       0.94      0.94      0.94       500

    accuracy                           0.94      1000
   macro avg       0.94      0.94      0.94      1000
weighted avg       0.94      0.94      0.94      1000



In [52]:
print("Test:")
print(classification_report(y_test, clf.predict(x_test)))

Test:
              precision    recall  f1-score   support

           0       0.84      0.93      0.88       500
           1       0.92      0.82      0.87       500

    accuracy                           0.88      1000
   macro avg       0.88      0.88      0.88      1000
weighted avg       0.88      0.88      0.88      1000



### 5) 5 documenti del test set dove il modello è più incerto in fase di classificazione

In [53]:
scores = clf.decision_function(x_test)

In [54]:
uncertainty = np.abs(scores)
most_uncertain_idx = np.argsort(uncertainty)[:5]

In [55]:
print(most_uncertain_idx)

[360 337 686 857  61]


In [56]:
for idx in most_uncertain_idx:
    print(f"Indice:{idx}")
    print(f"Score: {scores[idx]:.4f}")
    print(f"Label vera: {y_test[idx]}")
    print(f"Predizione: {clf.predict(x_test[idx].reshape(1,-1))[0]}")
    print("___")

Indice:360
Score: -0.0049
Label vera: 1
Predizione: 0
___
Indice:337
Score: -0.0067
Label vera: 0
Predizione: 0
___
Indice:686
Score: 0.0155
Label vera: 1
Predizione: 1
___
Indice:857
Score: 0.0330
Label vera: 0
Predizione: 1
___
Indice:61
Score: -0.0338
Label vera: 1
Predizione: 0
___


In [57]:
uncertain_docs = []

for idx in most_uncertain_idx:
    doc_id = test_ud.index[idx]
    pred = clf.predict(x_test[idx].reshape(1, -1))[0]
    
    uncertain_docs.append({
        "indice": int(idx),
        "score": round(float(scores[idx]), 4),
        "label_vera": int(y_test[idx]),
        "predizione": int(pred),
        "testo": df_test.loc[doc_id, "text"]
    })

uncertain_docs_df = pd.DataFrame(uncertain_docs)
uncertain_docs_df

,indice,score,label_vera,predizione,testo
0,360,-0.0049,1,0,"Una qualificazione che, francamente, lascia un..."
1,337,-0.0067,0,0,"Culle di pelliccia, quindi, amiche del respiro..."
2,686,0.0155,1,1,"Una cifra considerevole, che si somma a quelle..."
3,857,0.0330,0,1,Ma il trauma rimane. A prendere di mira la tur...
4,61,-0.0338,1,0,"According to Amnesty, sino al 2023 sarebbero s..."


### 6) Estrazione delle 20 features più rilevanti ai fini della classificazione

Coefficiente positivo -> classe 1: testo generato da AI


Coefficiente negativo -> classe 0: testo generato da umano

In [58]:
coef = clf.coef_[0]
top_20_idx = np.argsort(np.abs(coef))[::-1][:20]
top_20_features = [(common_cols[i], coef[i]) for i in top_20_idx]

In [59]:
for feature, c in top_20_features:
    print(f"{feature}: {c:.4f}")

n_tokens: -2.4203
dep_dist_advmod: 1.8717
upos_dist_ADV: -1.5223
lexical_density: -1.4563
n_prepositional_chains: 1.3106
ttr_form_chunks_200: -1.2966
dep_dist_det: -1.2358
upos_dist_NUM: -1.1195
upos_dist_PUNCT: 1.0152
upos_dist_PRON: -0.9997
upos_dist_DET: 0.9337
dep_dist_flat:foreign: -0.8925
dep_dist_case: -0.7970
upos_dist_VERB: -0.7753
avg_prepositional_chain_len: -0.7634
avg_subordinate_chain_len: -0.7145
upos_dist_ADP: 0.6032
upos_dist_PROPN: -0.5878
dep_dist_acl:relcl: 0.5427
dep_dist_nsubj: 0.5012
